# 資料集處理

本 notebook 用於準備中英文標題黨分類模型的訓練資料。

流程會讀取中文 WCD CSV 與英文 Webis-Clickbait-17 JSONL 檔案，將兩種資料轉成相同欄位格式，接著合併、打亂，並輸出 train / valid / test CSV 檔。

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# 讓 notebook 不管從專案 root 或 notebooks/ 裡執行，都能找到正確的專案根目錄。
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# 原始資料位置：中文 WCD 是 CSV，英文 Webis 解壓後是 JSONL + media 資料夾。
BASE_DATA_DIR = PROJECT_ROOT / "dataset" / "baseDataSet"
CHINESE_DATA_PATH = BASE_DATA_DIR / "all_labeled.csv"
WEBIS_DATA_DIR = BASE_DATA_DIR / "clickbait17-train-170630"

# 處理後資料會輸出到這裡，之後 baseline、transformer 訓練都讀這個資料夾。
PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"

# 中英文資料最後都會整理成同一組欄位，方便後面直接合併和訓練。
SCHEMA = ["id", "lang", "title", "content", "label", "source"]

print("Project root:", PROJECT_ROOT)
print("Chinese dataset:", CHINESE_DATA_PATH)
print("Webis dataset:", WEBIS_DATA_DIR)
print("Output dir:", PROCESSED_DIR)


Project root: D:\homework\NLP\Project
Chinese dataset: D:\homework\NLP\Project\dataset\baseDataSet\all_labeled.csv
Webis dataset: D:\homework\NLP\Project\dataset\baseDataSet\clickbait17-train-170630
Output dir: D:\homework\NLP\Project\dataset\processed


## 輔助函式

In [2]:
def normalize_text(value):
    """把空值轉成空字串，並去掉前後空白，避免 NaN 混進訓練資料。"""
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def prepare_chinese_dataframe(df):
    """將中文 WCD 資料轉成統一欄位格式。"""
    # WCD 必須有標題、內文與人工標籤；少任何一欄都不能繼續處理。
    required = {"TITLE", "CONTENT", "T_LABEL"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Chinese dataset missing columns: {sorted(missing)}")

    n = len(df)
    width = len(str(n))
    # WCD 的 T_LABEL 有 0/1/2：0 是非標題黨，1 和 2 都視為標題黨。
    result = pd.DataFrame({
        "id": [f"zh_{i:0{width}d}" for i in range(n)],
        "lang": "zh",
        "title": df["TITLE"].map(normalize_text),
        "content": df["CONTENT"].map(normalize_text),
        "label": df["T_LABEL"].map(lambda x: 0 if int(x) == 0 else 1),
        "source": "wcd",
    })
    return result[SCHEMA]


def read_jsonl(path):
    """讀取 JSONL 檔案；每一行是一筆 JSON record。"""
    records = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def first_text(values):
    """Webis 的 postText 是 list，這裡取第一個文字作為 tweet/title。"""
    if isinstance(values, list) and values:
        return normalize_text(values[0])
    return ""


def join_paragraphs(values):
    """把 Webis 的多段文章內文合併成一個 content 欄位。"""
    if not isinstance(values, list):
        return ""
    cleaned = [normalize_text(v) for v in values]
    cleaned = [v for v in cleaned if v]
    return "\n\n".join(cleaned)


def prepare_english_records(instances, truth_records):
    """合併 Webis instances 與 truth labels，並轉成統一欄位格式。"""
    # truth.jsonl 只有標籤，instances.jsonl 只有文章內容；兩者靠 id 對應。
    truth_by_id = {str(item["id"]): item for item in truth_records}
    rows = []
    counter = 0

    for item in instances:
        item_id = str(item["id"])
        truth = truth_by_id.get(item_id)
        if truth is None:
            # 如果某筆文章找不到標籤，不能拿來做 supervised training。
            continue

        # 優先使用 postText 作為模型輸入標題；如果 postText 空白，就退回 targetTitle。
        title = first_text(item.get("postText")) or normalize_text(item.get("targetTitle"))
        content = join_paragraphs(item.get("targetParagraphs"))

        # Webis 的 truthMean 是 0 到 1 的分數，這裡用 0.5 當二元分類門檻。
        label = 1 if float(truth["truthMean"]) >= 0.5 else 0

        rows.append({
            "id": f"en_{counter}",  # 暫用 counter，稍後補零
            "lang": "en",
            "title": title,
            "content": content,
            "label": label,
            "source": "webis-clickbait-17",
        })
        counter += 1

    width = len(str(counter))
    df = pd.DataFrame(rows, columns=SCHEMA)
    df["id"] = [f"en_{i:0{width}d}" for i in range(len(df))]
    return df


def load_webis_folder(folder):
    """從 Webis 解壓資料夾讀取 instances.jsonl 和 truth.jsonl。"""
    folder = Path(folder)
    instances = read_jsonl(folder / "instances.jsonl")
    truth = read_jsonl(folder / "truth.jsonl")
    return prepare_english_records(instances, truth)


def combine_datasets(*frames):
    """合併多個已經轉成 SCHEMA 的資料表。"""
    combined = pd.concat(frames, ignore_index=True)
    combined = combined[SCHEMA].copy()
    combined["label"] = combined["label"].astype(int)

    # 空標題無法做標題黨偵測；label 也只保留 0/1。
    combined = combined[(combined["title"].str.len() > 0) & (combined["label"].isin([0, 1]))]
    return combined.reset_index(drop=True)


def split_dataset(df, train_ratio=0.8, valid_ratio=0.1, seed=42):
    """依照 lang + label 分層切成 train / valid / test。"""
    if train_ratio <= 0 or valid_ratio < 0 or train_ratio + valid_ratio >= 1:
        raise ValueError("Ratios must satisfy train_ratio > 0, valid_ratio >= 0, and train + valid < 1.")

    stratify_key = df["lang"].astype(str) + "_" + df["label"].astype(str)
    temp_ratio = 1.0 - train_ratio

    train, temp = train_test_split(
        df,
        test_size=temp_ratio,
        random_state=seed,
        shuffle=True,
        stratify=stratify_key,
    )

    temp_stratify_key = temp["lang"].astype(str) + "_" + temp["label"].astype(str)
    test_ratio_within_temp = (1.0 - train_ratio - valid_ratio) / temp_ratio
    valid, test = train_test_split(
        temp,
        test_size=test_ratio_within_temp,
        random_state=seed,
        shuffle=True,
        stratify=temp_stratify_key,
    )

    train = train.reset_index(drop=True)
    valid = valid.reset_index(drop=True)
    test = test.reset_index(drop=True)
    return train, valid, test


def save_datasets(combined, train, valid, test, output_dir):
    """輸出完整資料與三個切分資料。"""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # utf-8-sig 讓 Windows Excel 開啟中文時比較不容易亂碼。
    combined.to_csv(output_dir / "unified_clickbait.csv", index=False, encoding="utf-8-sig")
    train.to_csv(output_dir / "train.csv", index=False, encoding="utf-8-sig")
    valid.to_csv(output_dir / "valid.csv", index=False, encoding="utf-8-sig")
    test.to_csv(output_dir / "test.csv", index=False, encoding="utf-8-sig")


def summarize(df, name):
    """印出資料筆數、語言分布和 label 分布，方便快速檢查資料是否合理。"""
    print(f"\n{name}: {len(df):,} rows")
    print("language counts:")
    print(df["lang"].value_counts().to_string())
    print("label counts:")
    print(df["label"].value_counts().sort_index().to_string())


## 基本測試

In [3]:
# 測試中文 label 轉換：T_LABEL=0 保持 0，T_LABEL=1/2 都合併成 1。
sample_cn = pd.DataFrame([
    {"TITLE": "普通新聞", "CONTENT": "這是一則普通新聞", "T_LABEL": 0},
    {"TITLE": "震驚！你一定想不到", "CONTENT": "文章內容", "T_LABEL": 1},
    {"TITLE": "結果竟然是這樣", "CONTENT": "文章內容", "T_LABEL": 2},
])
cn = prepare_chinese_dataframe(sample_cn)

assert list(cn.columns) == SCHEMA
assert cn["label"].tolist() == [0, 1, 1]
assert cn["lang"].unique().tolist() == ["zh"]
assert cn["id"].tolist() == ["zh_0", "zh_1", "zh_2"]  # 3 筆，1 位數，無補零


In [4]:
# 測試英文 Webis 合併邏輯：instances 提供文字，truth 提供標籤，兩者用 id 對齊。
sample_instances = [
    {"id": "a", "postText": ["Click me"], "targetTitle": "Target A", "targetParagraphs": ["Paragraph A1", "Paragraph A2"]},
    {"id": "b", "postText": [], "targetTitle": "Target B", "targetParagraphs": []},
]
sample_truth = [
    {"id": "a", "truthMean": 0.7, "truthClass": "clickbait"},
    {"id": "b", "truthMean": 0.2, "truthClass": "no-clickbait"},
]
en = prepare_english_records(sample_instances, sample_truth)

assert list(en.columns) == SCHEMA
assert en["title"].tolist() == ["Click me", "Target B"]
assert en["content"].tolist() == ["Paragraph A1\n\nParagraph A2", ""]
assert en["label"].tolist() == [1, 0]
assert en["id"].tolist() == ["en_0", "en_1"]  # 2 筆，1 位數，無補零


In [5]:
# 測試合併與切分：確認資料筆數沒有遺失，且 lang + label 分層比例被保留下來。
combined_sample = combine_datasets(cn, en)
assert len(combined_sample) == 5
assert set(combined_sample["lang"]) == {"zh", "en"}
assert set(combined_sample["label"]) == {0, 1}

stratified_sample = pd.DataFrame([
    {
        "id": f"{lang}_{label}_{i}",
        "lang": lang,
        "title": f"sample {lang} {label} {i}",
        "content": "sample content",
        "label": label,
        "source": "test",
    }
    for lang in ["en", "zh"]
    for label in [0, 1]
    for i in range(10)
])
train_sample, valid_sample, test_sample = split_dataset(stratified_sample, seed=42)

assert len(train_sample) == 32
assert len(valid_sample) == 4
assert len(test_sample) == 4
for split_df, expected_count in [(train_sample, 8), (valid_sample, 1), (test_sample, 1)]:
    group_counts = split_df.groupby(["lang", "label"]).size()
    assert (group_counts == expected_count).all()

print("Smoke tests passed.")


Smoke tests passed.


## 建立資料集檔案

In [6]:
# 讀取完整中文資料，並轉成統一格式。
chinese_raw = pd.read_csv(CHINESE_DATA_PATH)
chinese_df = prepare_chinese_dataframe(chinese_raw)

# 讀取完整英文 Webis 資料，並轉成統一格式。
english_df = load_webis_folder(WEBIS_DATA_DIR)

# 合併中英文資料，再切成 80% train、10% valid、10% test。
combined_df = combine_datasets(chinese_df, english_df)
train_df, valid_df, test_df = split_dataset(combined_df, train_ratio=0.8, valid_ratio=0.1, seed=42)

# 寫出 CSV，給後續 baseline 與 transformer training 使用。
save_datasets(combined_df, train_df, valid_df, test_df, PROCESSED_DIR)

# 印出摘要，檢查資料量與 label 分布。
summarize(chinese_df, "Chinese WCD")
summarize(english_df, "English Webis")
summarize(combined_df, "Unified dataset")
summarize(train_df, "Train")
summarize(valid_df, "Valid")
summarize(test_df, "Test")



Chinese WCD: 16,656 rows
language counts:
lang
zh    16656
label counts:
label
0     4731
1    11925

English Webis: 19,538 rows
language counts:
lang
en    19538
label counts:
label
0    14798
1     4740

Unified dataset: 36,194 rows
language counts:
lang
en    19538
zh    16656
label counts:
label
0    19529
1    16665

Train: 28,955 rows
language counts:
lang
en    15630
zh    13325
label counts:
label
0    15623
1    13332

Valid: 3,619 rows
language counts:
lang
en    1954
zh    1665
label counts:
label
0    1953
1    1666

Test: 3,620 rows
language counts:
lang
en    1954
zh    1666
label counts:
label
0    1953
1    1667


In [7]:
# 最後檢查輸出檔是否存在，且切分後筆數加總等於完整資料筆數。
expected_files = ["unified_clickbait.csv", "train.csv", "valid.csv", "test.csv"]
for filename in expected_files:
    path = PROCESSED_DIR / filename
    assert path.exists(), f"Missing output file: {path}"

assert len(train_df) + len(valid_df) + len(test_df) == len(combined_df)
assert list(combined_df.columns) == SCHEMA
assert set(combined_df["label"].unique()).issubset({0, 1})
assert {"zh", "en"}.issubset(set(combined_df["lang"].unique()))

print("Dataset files written to:", PROCESSED_DIR)


Dataset files written to: D:\homework\NLP\Project\dataset\processed
